In [1]:
from Langchain_learning_2 import prompt

pip
install
langgraph - cli
jupyter - i
https: // mirrors.aliyun.com / pypi / simple /

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
  Using cached https://mirrors.aliyun.com/pypi/packages/93/38/6d7231122820bded2467dd88028e2ff2e2210a166f18311ea261ee5998f8/langgraph_cli-0.4.21-py3-none-any.whl (73 kB)
  Using cached https://mirrors.aliyun.com/pypi/packages/38/64/285f20a31679bf547b75602702f7800e74dbabae36ef324f716c02804753/jupyter-1.1.1-py2.py3-none-any.whl (2.7 kB)
  Using cached https://mirrors.aliyun.com/pypi/packages/ca/77/71d78d58f15c22db16328a476426f7ac4a60d3a5a7ba3b9627ee2f7903d4/jupyter_console-6.6.3-py3-none-any.whl (24 kB)
  Using cached https://mirrors.aliyun.com/pypi/packages/56/6d/0d9848617b9f753b87f214f1c682592f7ca42de085f564352f10f0843026/ipywidgets-8.1.8-py3-none-any.whl (139 kB)
     ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
     ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
     --------------------------------- ------ 1.8/2.2 MB 5.0 MB/s eta 0:00:01
     ---------------------------------------- 2.

In [3]:
pip
install
langgraph
langchain
langchain - openai
python - dotenv - i
https: // mirrors.aliyun.com / pypi / simple /


Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.


In [4]:
from importlib.metadata import version

print(f"LangGraph 版本: {version('langgraph')}")

LangGraph 版本: 1.1.6


## LangGraph的学习代码
* Python相比Java有这天然的逻辑清晰度
* 定义状态,节点之间流转的数据包,实现上下节点之间的通信
* 创建节点(Python函数)
* 添加边(流程)
* 实现输出

In [5]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, Optional
from langgraph.graph.message import add_messages


# Step 1: 定义 State
class SimpleState(TypedDict):
    # 消息历史（add_messages reducer 自动追加而非覆盖）
    messages: Annotated[list, add_messages]
    processed: bool


# Step 2: 定义节点函数
def greet_node(state: SimpleState) -> dict:
    """欢迎节点：生成问候语"""
    print(f"[greet_node] 收到消息: {state['message']}")
    return {"message": f"你好！{state['message']}"}


def process_node(state: SimpleState) -> dict:
    """处理节点：标记为已处理"""
    print(f"[process_node] 处理消息: {state['message']}")
    return {"processed": True}


# Step 3: 构建图
builder = StateGraph(SimpleState)

# 添加节点 创建节点每个节点也就是Python函数
builder.add_node("greet", greet_node)
builder.add_node("process", process_node)

# 添加边 START | GREET | PROCESS | END
builder.add_edge(START, "greet")
builder.add_edge("greet", "process")
builder.add_edge("process", END)

# Step 4: 编译图
graph = builder.compile()

# Step 5: 运行
result = graph.invoke({
    "message": "世界",
    "processed": False
})

print(f"\n最终结果: {result}")

[greet_node] 收到消息: 世界
[process_node] 处理消息: 你好！世界

最终结果: {'message': '你好！世界', 'processed': True}


In [8]:
# 在 Jupyter Notebook 中可视化
from IPython.display import Image

Image(graph.get_graph().draw_mermaid_png())

# 或者打印 Mermaid 格式
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	greet(greet)
	process(process)
	__end__([<p>__end__</p>]):::last
	__start__ --> greet;
	greet --> process;
	process --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



**使用mermaid语言的代码块直接展示结果**
```mermaid
graph TD
    __start__([__start__]):::first
    greet(greet)
    process(process)
    __end__([__end__]):::last
    __start__ --> greet;
    greet --> process;
    process --> __end__;
    classDef default fill:#f2f0ff,line-height:1.2
    classDef first fill-opacity:0
    classDef last fill:#bfb6fc
```

In [9]:
# 定义状态
# 在生产环境中使用pydantic

from pydantic import BaseModel, Field
from typing import Annotated
from langgraph.graph.message import add_messages


class ProductionState(BaseModel):
    # messages用于存储对话的消息列表
    # add_messages多节点追加信息
    # default_factory=list 表示每次创建一个新的 ProductionState 实例时，messages 会自动初始化为一个空列表并非共享
    messages: Annotated[list, add_messages] = Field(default_factory=list)
    user_id: str = ""
    confidence_score: float = 0.0

    # 设置 arbitrary_types_allowed = True 后，Pydantic 允许任意 Python 类型作为字段值，只要在运行时能正确处理即可
    # 原本只默许JSON友好型文本结构
    class Config:
        arbitrary_types_allowed = True

C:\Users\DELL\AppData\Local\Temp\ipykernel_33644\158366669.py:7: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class ProductionState(BaseModel):


## 节点的定义
节点的本质就是一个"函数"
节点就是一个接收状态、返回状态更新的纯函数

类通过`__call__`实现函数调用
大模型通过模型自主处理(提示词)返回消息
普通节点就是函数, 注意状态的改变

* 普通节点
* 大模型节点
* 类节点
* 异步节点

普通节点:
* 读取状态
* 执行操作
* 返回消息

大模型节点:
* 编写提示词
* 提示词和历史消息(上下文)的拼接
* 调用模型

异步节点:
* 实行大量IO操作
*

In [10]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage

load_dotenv()

# 使用 DeepSeek 模型
llm = ChatOpenAI(
    model=os.getenv('DEEPSEEK_MODEL', 'deepseek-chat'),
    openai_api_key=os.getenv('DEEPSEEK_API_KEY'),
    openai_api_base=os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com'),
    temperature=0
)


def llm_node(state: dict) -> dict:
    """调用 LLM 的节点"""
    # 使用LangGraph规定的消息类型
    system_prompt = SystemMessage(content="你是一个有帮助的助手。")

    # 将系统提示与对话历史合并
    messages = [system_prompt] + state["messages"]

    # 调用 LLM
    response = llm.invoke(messages)

    return {"messages": [response]}



In [ ]:
# 普通节点
def simple_node(state: SimpleState) -> dict:
    # 读取状态
    last_message = state["messages"][-1]

    # 执行操作
    response = f"收到: {last_message.content}"

    # 返回部分状态更新
    return {
        "messages": [{"role": "assistant", "content": response}]
    }

In [ ]:
import asyncio


# 实现异步节点
async def async_node(state: SimpleState) -> dict:
    """异步节点，适合 I/O 密集型操作"""
    # 模拟异步操作（如 API 调用、数据库查询）
    await asyncio.sleep(0.1)

    result = await some_async_api_call(state["messages"][-1].content)
    return {"messages": [{"role": "assistant", "content": result}]}


# 使用异步图
result = await graph.ainvoke({"messages": [...]})

In [ ]:

# 使用类作为节点
class RouterNode:
    # 构造器
    def __init__(self, llm, system_prompt: str):
        self.llm = llm
        self.system_prompt = system_prompt

    # 实现函数式调用
    def __call__(self, state: SimpleState) -> dict:
        """类实例可以作为节点使用"""
        # 拼接系统提示词和历史信息
        messages = [
            SystemMessage(content=self.system_prompt),
            # 解包历史对话信息
            *state["messages"]
        ]
        response = self.llm.invoke(messages)
        return {"messages": [response]}


# 添加类节点
router = RouterNode(llm, "你是一个专业的路由助手。")
builder.add_node("router", router)

In [ ]:
# 固定路径：node_a 完成后始终执行 node_b
builder.add_edge("node_a", "node_b")

# 结束：node_a 完成后图结束
builder.add_edge("node_a", END)

In [ ]:
def route_after_llm(state: SimpleState) -> str:
    """
    路由函数：根据 LLM 的最新输出决定走哪条路径
    返回值必须是已注册节点名称或 END
    """
    last_message = state["messages"][-1]

    # 如果 LLM 请求使用工具
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    # 否则结束
    return END


# 添加条件边
builder.add_conditional_edges(
    "llm",  # 源节点
    route_after_llm,  # 路由函数
    {
        "tools": "tool_executor",  # 返回 "tools" 时 -> tool_executor 节点
        END: END  # 返回 END 时 -> 结束
    }
)

In [ ]:
# 从一个节点并行分叉到多个节点
builder.add_edge("start_node", "branch_a")
builder.add_edge("start_node", "branch_b")
builder.add_edge("start_node", "branch_c")

# 多个节点汇聚到一个节点（Fan-in）
builder.add_edge("branch_a", "merge_node")
builder.add_edge("branch_b", "merge_node")
builder.add_edge("branch_c", "merge_node")

In [11]:

# 实现条件路由

import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

# 初始化 LLM
llm = ChatOpenAI(
    model=os.getenv('DEEPSEEK_MODEL', 'deepseek-chat'),
    openai_api_key=os.getenv('DEEPSEEK_API_KEY'),
    openai_api_base=os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com'),
    temperature=0.7
)


# 将反复操作的内容封装为类函数调用
class LLM_Invoke:
    def __init__(self, llm, prompt: str):
        self.llm = llm
        self.prompt = prompt

    # state 参数由 LangGraph 自动传入
    def __call__(self, state: SimpleState) -> dict:
        message = [
            SystemMessage(content=self.prompt),
            *state["messages"]
        ]
        response = self.llm.invoke(message)
        return {"messages": [response]}


# 路由函数
def classify_intent(state: MessagesState) -> str:
    """根据用户意图路由到不同的 Agent"""
    last_message = state["messages"][-1]
    content = last_message.content.lower()

    if "天气" in content or "温度" in content:
        return "weather_agent"
    elif "代码" in content or "编程" in content:
        return "code_agent"
    elif "再见" in content or "退出" in content:
        return "farewell"
    else:
        return "general_agent"


# 定义各个 Agent 节点
def router_node(state: MessagesState) -> dict:
    """路由节点：不做处理，只用于触发路由判断"""
    return {}


def weather_node(state: MessagesState) -> dict:
    """天气 Agent"""

    return LLM_Invoke(llm, "你是一个天气助手,给出准确的天气预报")


def code_node(state: MessagesState) -> dict:
    """代码 Agent"""

    return LLM_Invoke(llm, "你是一个编程助手，擅长解答代码问题并给出清晰的代码示例。")


def general_node(state: MessagesState) -> dict:
    """通用 Agent"""

    return LLM_Invoke(llm, "你是一个友善的 AI 助手，可以回答各种问题。")


def farewell_node(state: MessagesState) -> dict:
    """告别节点"""
    return {"messages": [{"role": "assistant", "content": "再见！期待下次与你交流。"}]}


# 构建图
builder = StateGraph(MessagesState)

# 添加节点
builder.add_node("router", router_node)
builder.add_node("weather_agent", weather_node)
builder.add_node("code_agent", code_node)
builder.add_node("general_agent", general_node)
builder.add_node("farewell", farewell_node)

# 添加边
builder.add_edge(START, "router")
builder.add_conditional_edges(
    "router",
    classify_intent,
    {
        "weather_agent": "weather_agent",
        "code_agent": "code_agent",
        "general_agent": "general_agent",
        "farewell": "farewell",
    }
)

# 所有 agent 节点处理完后结束
for node in ["weather_agent", "code_agent", "general_agent", "farewell"]:
    builder.add_edge(node, END)

# 编译图
graph = builder.compile()

# 测试不同意图
test_inputs = [
    "北京今天天气怎么样？",
    "帮我写一个 Python 快速排序",
    "你好，介绍一下你自己",
    "再见啦！"
]

for user_input in test_inputs:
    print(f"\n用户: {user_input}")
    result = graph.invoke({"messages": [HumanMessage(content=user_input)]})
    print(f"助手: {result['messages'][-1].content[:100]}...")
    print("-" * 50)


用户: 北京今天天气怎么样？


InvalidUpdateError: Expected dict, got <__main__.LLM_Invoke object at 0x000001FB916B5250>
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE

In [2]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

# 初始化 LLM（使用 DeepSeek）
llm = ChatOpenAI(
    model=os.getenv('DEEPSEEK_MODEL', 'deepseek-chat'),
    openai_api_key=os.getenv('DEEPSEEK_API_KEY'),
    openai_api_base=os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com'),
    temperature=0.7
)

SYSTEM_PROMPT = """你是一个友善、专业的 AI 助手。
你的回答应该：
- 简洁清晰
- 使用中文回复
- 在不确定时主动询问用户
"""


def chatbot_node(state: MessagesState) -> dict:
    """核心对话节点"""
    system = SystemMessage(content=SYSTEM_PROMPT)
    messages = [system] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


# 构建图
builder = StateGraph(MessagesState)
# 添加节点 (别名,节点名)
builder.add_node("chatbot", chatbot_node)

# 添加边
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()


# 多轮对话函数
def chat(conversation_history: list, user_input: str) -> tuple[str, list]:
    """处理单轮对话，返回 AI 响应和更新后的历史"""

    # 历史信息插入人类问题
    conversation_history.append(HumanMessage(content=user_input))

    # 执行结果
    result = graph.invoke({"messages": conversation_history})

    # 添加结果
    conversation_history = result["messages"]

    # 取出最后加入的信息
    ai_response = conversation_history[-1].content

    return ai_response, conversation_history


# 多轮对话示例
history = []
while True:
    user_input = input("你: ")
    if user_input.lower() in ["退出", "exit", "quit"]:
        print("再见！")
        break

    response, history = chat(history, user_input)
    print(f"助手: {response}\n")

助手: （温柔地笑着放下包）今天项目临时加了会议，让你担心啦。锅里给你留了汤，要现在喝吗？

助手: （轻轻拉住你的手）是我不好，下次一定早点回来。给你带了街角那家甜品店的提拉米苏，现在想吃吗？

助手: （后退半步露出困惑的表情）抱歉，我可能走错楼层了。您住几零几？我这就离开。

助手: （站在门边礼貌地微笑）您可能认错人了，我是新搬来三楼的住户。需要帮您联系家人吗？

助手: （神色担忧地拿出手机）女士，您似乎有些混乱。需要我帮您拨打社区求助电话吗？您家人的联系方式方便告诉我吗？

再见！


In [ ]:
# 实现条件边

def route_after_llm(state: AgentState) -> str:
    """
    路由函数：根据 LLM 的最新输出决定走哪条路径
    返回值必须是已注册节点名称或 END
    """
    last_message = state["messages"][-1]

    # 如果 LLM 请求使用工具
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    # 否则结束
    return END


# 添加条件边
builder.add_conditional_edges(
    "llm",  # 源节点
    route_after_llm,  # 路由函数
    {
        "tools": "tool_executor",  # 返回 "tools" 时 -> tool_executor 节点
        END: END  # 返回 END 时 -> 结束
    }
)

## LangGraph设计
节点设计:
* 开始节点
* 用户输入
* llm大模型判断是否需要RAG
    * 进入法务Agent系统
    * 普通chat
* RAG查询节点
* llm上下文拼接后给出建议
* 输出节点

工具设计
* 在线搜索引擎
* PDF输出,llm不直接写入PDF,由MarkDown渲染成Html
* 长文本摘要工具,当用户的问题过长先一步压缩问题(关键字,语义)
* 邮箱发送工具,需要提前获取用户的邮件
* Python代码沙盒
* MarkDown转其他格式文本(docx,pdf,txt)


遵从MVP开发原则:
开发优先级:
* 提问 - 检索 - llm - 输出
* RAG知识库
* 数据库开发
* 检索系统BM25
* 文本切分



In [8]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field,ConfigDict
from typing import Annotated
from langgraph.graph.message import add_messages

load_dotenv()

# 初始化 LLM
llm = ChatOpenAI(
    model=os.getenv('DEEPSEEK_MODEL', 'deepseek-chat'),
    openai_api_key=os.getenv('DEEPSEEK_API_KEY'),
    openai_api_base=os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com'),
    temperature=0.3
)


class AgentState(BaseModel):
    # 消息列表，支持追加消息
    messages: Annotated[list, add_messages] = Field(default_factory=list)
    # 用户id
    user_id: str = ""
    # 置信度
    confidence_score: float = 0.0

    model_config = ConfigDict(
        arbitrary_types_allowed=True,
    )


# 设计类 实现llm的调用
class LLM_Invoke:
    def __init__(self, llm, prompt: str):
        self.llm = llm
        self.prompt = prompt

    # state 参数由 LangGraph 自动传入
    def __call__(self, state: AgentState) -> dict:
        message = [
            SystemMessage(content=self.prompt),
            *state.messages
        ]
        response = self.llm.invoke(message)
        return {"messages": [response]}


# 设置简单的大模型询问节点
def Simple_llm_node(state: AgentState) -> dict:
    SYSTEM_PROMPT = """
    你是一个智能、可靠、友善的AI助手。请遵循以下原则：

    1. 准确优先：不确定时不编造，明确说“我不知道”或“需要进一步信息”。
    2. 简洁清晰：直接回答问题核心，避免啰嗦和无关内容。
    3. 安全中立：不生成暴力、色情、仇恨或歧视性内容；不提供违法建议。
    4. 结构友好：当需要分点、列表或步骤时，使用清晰的结构（如序号或短横）。
    5. 仅输出回答内容：不要添加“作为AI模型…”之类的开场白或免责声明，除非用户明确询问你的身份。
    """
    # 创建实例,注意参数的顺序
    llm_invoke = LLM_Invoke(llm, SYSTEM_PROMPT)
    # 实例化后传入state,得到dict结果
    response = llm_invoke(state)
    return response


builder = StateGraph(AgentState)

# 添加节点
builder.add_node("Simple_chat_node", Simple_llm_node)

builder.add_edge(START, "Simple_chat_node")
builder.add_edge("Simple_chat_node", END)

# 编译图
graph = builder.compile()

response = graph.invoke({"messages": [HumanMessage(content="泥嚎,我有点累了")]})


reply = response['messages'][-1].content
usage = response['messages'][-1].usage_metadata


print(f"AI回答:{reply}")
print(f"AI用量:{usage}")













C:\Users\DELL\AppData\Local\Temp\ipykernel_30764\3078279818.py:22: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class AgentState(BaseModel):


{'messages': [HumanMessage(content='泥嚎,我有点累了', additional_kwargs={}, response_metadata={}, id='b0b368bd-494a-42b7-86e3-52ffc0a6ab4b'), AIMessage(content='累了就好好休息一下吧，喝点水，闭上眼睛放松几分钟。如果需要，可以听听舒缓的音乐或者简单活动一下身体。照顾好自己最重要。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 152, 'total_tokens': 182, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 24}, 'model_provider': 'openai', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': '82eca8e0-52cc-4307-a0a3-b81634a4d776', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9919-a973-7d52-896a-dee7612d414f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 30, 'total_tokens': 182, 'input_token_details': {'cache_read': 128}, 'output_token_details': {}})]

In [ ]:
{'messages': [HumanMessage(content='泥嚎,我有点累了', additional_kwargs={}, response_metadata={},
                           id='4a2f9d7c-ab78-4252-b85f-131ab3623938'),
              AIMessage(content='累了就好好休息一下吧，喝点水，放松几分钟。如果需要，可以试试深呼吸或者短暂闭目养神。',
                        additional_kwargs={'refusal': None}, response_metadata={
                      'token_usage': {'completion_tokens': 25, 'prompt_tokens': 152, 'total_tokens': 177,
                                      'completion_tokens_details': None,
                                      'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128},
                                      'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 24},
                      'model_provider': 'openai', 'model_name': 'deepseek-chat',
                      'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410',
                      'id': '949759ee-7f9c-42a2-91da-3aa06cd2515f', 'finish_reason': 'stop', 'logprobs': None},
                        id='lc_run--019d9914-c854-7c93-b6a5-001e69079532-0', tool_calls=[], invalid_tool_calls=[],
                        usage_metadata={'input_tokens': 152, 'output_tokens': 25, 'total_tokens': 177,
                                        'input_token_details': {'cache_read': 128}, 'output_token_details': {}})]}

In [9]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

# 初始化 LLM
llm = ChatOpenAI(
    model=os.getenv('DEEPSEEK_MODEL', 'deepseek-chat'),
    openai_api_key=os.getenv('DEEPSEEK_API_KEY'),
    openai_api_base=os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com'),
    temperature=0.7
)

# 路由函数
def classify_intent(state: MessagesState) -> str:
    """根据用户意图路由到不同的 Agent"""
    last_message = state["messages"][-1]
    content = last_message.content.lower()

    if "天气" in content or "温度" in content:
        return "weather_agent"
    elif "代码" in content or "编程" in content:
        return "code_agent"
    elif "再见" in content or "退出" in content:
        return "farewell"
    else:
        return "general_agent"

# 定义各个 Agent 节点
def router_node(state: MessagesState) -> dict:
    """路由节点：不做处理，只用于触发路由判断"""
    return {}

def weather_node(state: MessagesState) -> dict:
    """天气 Agent"""
    response = llm.invoke([
        SystemMessage(content="你是一个天气助手，友好地回答天气相关问题。如果没有实时数据，可以给出一般性建议。"),
        *state["messages"]
    ])
    return {"messages": [response]}

def code_node(state: MessagesState) -> dict:
    """代码 Agent"""
    response = llm.invoke([
        SystemMessage(content="你是一个编程助手，擅长解答代码问题并给出清晰的代码示例。"),
        *state["messages"]
    ])
    return {"messages": [response]}

def general_node(state: MessagesState) -> dict:
    """通用 Agent"""
    response = llm.invoke([
        SystemMessage(content="你是一个友善的 AI 助手，可以回答各种问题。"),
        *state["messages"]
    ])
    return {"messages": [response]}

def farewell_node(state: MessagesState) -> dict:
    """告别节点"""
    return {"messages": [{"role": "assistant", "content": "再见！期待下次与你交流。"}]}

# 构建图
builder = StateGraph(MessagesState)

# 添加节点
builder.add_node("router", router_node)
builder.add_node("weather_agent", weather_node)
builder.add_node("code_agent", code_node)
builder.add_node("general_agent", general_node)
builder.add_node("farewell", farewell_node)

# 添加边
builder.add_edge(START, "router")
builder.add_conditional_edges(
    "router",
    classify_intent,
    {
        "weather_agent": "weather_agent",
        "code_agent": "code_agent",
        "general_agent": "general_agent",
        "farewell": "farewell",
    }
)

# 所有 agent 节点处理完后结束
for node in ["weather_agent", "code_agent", "general_agent", "farewell"]:
    builder.add_edge(node, END)

# 编译图
graph = builder.compile()

# 测试不同意图
test_inputs = [
    "北京今天天气怎么样？",
    "帮我写一个 Python 快速排序",
    "你好，介绍一下你自己",
    "再见啦！"
]

for user_input in test_inputs:
    print(f"\n用户: {user_input}")
    result = graph.invoke({"messages": [HumanMessage(content=user_input)]})
    print(f"助手: {result['messages'][-1].content[:100]}...")
    print("-" * 50)


用户: 北京今天天气怎么样？
助手: 很抱歉，我无法提供实时天气数据。建议您通过手机天气应用、网站（如中国天气网）或搜索引擎查询北京今天的天气情况，以获取最准确的温度、湿度及降水概率等信息。出行前记得关注实时更新哦！ 🌤️...
--------------------------------------------------

用户: 帮我写一个 Python 快速排序
助手: 我来帮你写一个快速排序的 Python 实现：

```python
def quick_sort(arr):
    """
    快速排序算法
    """
    # 递归终止条件：数组为空...
--------------------------------------------------

用户: 你好，介绍一下你自己
助手: 你好！我是DeepSeek，一个由深度求索公司创造的AI助手。很高兴认识你！😊

让我简单介绍一下自己：

**我的特点：**
- 我是一个纯文本模型，擅长理解和生成自然语言
- 知识截止到2024年...
--------------------------------------------------

用户: 再见啦！
助手: 再见！期待下次与你交流。...
--------------------------------------------------


In [1]:
pip install langgraph-cli -i https://mirrors.aliyun.com/pypi/simple/

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install langgraph-cli -i https://mirrors.aliyun.com/pypi/simple/

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.
